In [8]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# download .env
load_dotenv(".env")

try:
    HF_TOKEN= os.getenv('TOKEN_HF')
    # добовляю для NeMo Guardrails
    OPENAI_API_KEY= os.getenv('OPENAI_API_KEY')

    if HF_TOKEN is None:
        raise ValueError("TOKEN_HF не найдена!")

    if OPENAI_API_KEY is None:
        raise ValueError("OPENAI_API_KEY не найдена!")

    login(HF_TOKEN)
    print("hugging_face API key successful installed!")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("OpenAI API key successful installed!")

except Exception as e:
    print(f"Ошибка: {e}")

hugging_face API key successful installed!
OpenAI API key successful installed!


In [9]:
!pip install llama-index pymupdf openai

In [ ]:
# модель 

In [1]:
MODEL_NAME = "IlyaGusev/saiga_llama3_8b"

In [2]:
import torch
from torch import bfloat16
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig, pipeline
import bitsandbytes


# 4-разрядная конфигурация для загрузки LLM с меньшим объемом памяти графического процессора
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                                            # Использую 4-битную квантизацию
    bnb_4bit_quant_type='nf4',                                    # используем формат NF4
    bnb_4bit_use_double_quant=True,                               # применить повторное квантовани
    bnb_4bit_compute_dtype=bfloat16                               # Тип из которого преобразуем (доступная точность модели)
)

# Загружаем модель с квантизацией
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,                      # параметры квантования
    torch_dtype=torch.float16,                                    # тип данных
    device_map="auto"                                             # автоматический выбор типа устройства
)

model.eval()

#Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Создание конфигурации генерации
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == 'system':
            prompt += f"<s>{message.role}\n{message.content}</s>\n"
        elif message.role == 'user':
            prompt += f"<s>{message.role}\n{message.content}</s>\n"
        elif message.role == 'bot':
            prompt += f"<s>bot\n"

    # ensure we start with a system prompt, insert blank if needed
    if not prompt.startswith("<s>system\n"):
        prompt = "<s>system\n</s>\n" + prompt

    # add final assistant prompt
    prompt = prompt + "<s>bot\n"
    return prompt

def completion_to_prompt(completion):
    return f"<s>system\n</s>\n<s>user\n{completion}</s>\n<s>bot\n"

In [4]:
from llama_index.llms.huggingface import HuggingFaceLLM

llm = HuggingFaceLLM(
    model=model,             # модель
    model_name=MODEL_NAME,   # идентификатор модели
    tokenizer=tokenizer,     # токенизатор
    max_new_tokens=generation_config.max_new_tokens, # параметр необходимо использовать здесь, и не использовать в generate_kwargs, иначе ошибка двойного использования
    model_kwargs={"quantization_config": quantization_config}, # параметры квантования
    generate_kwargs = {   # параметры для инференса
      "bos_token_id": generation_config.bos_token_id, # токен начала последовательности
      "eos_token_id": generation_config.eos_token_id, # токен окончания последовательности
      "pad_token_id": generation_config.pad_token_id, # токен пакетной обработки (указывает, что последовательность ещё не завершена)
      "no_repeat_ngram_size": generation_config.no_repeat_ngram_size,
      "repetition_penalty": generation_config.repetition_penalty,
      "temperature": generation_config.temperature,
      "do_sample": True,
      "top_k": 50,
      "top_p": 0.95
    },
    messages_to_prompt=messages_to_prompt,     # функция для преобразования сообщений к внутреннему формату
    completion_to_prompt=completion_to_prompt, # функции для генерации текста
    device_map="auto",                         # автоматически определять устройство
)

In [ ]:
!gdown "https://drive.google.com/uc?export=download&id=1QQEday3YxXd8b9mcDzlpPkI2i8W75Cvn"

In [ ]:
!unzip "./library.zip"

In [2]:
from llama_index.core import SimpleDirectoryReader

#Загрузка всех Pdf из папки
Documents = SimpleDirectoryReader("./library").load_data()

In [ ]:
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain_community.embeddings import HuggingFaceEmbeddings

embed_model = LangchainEmbedding(
  HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
)

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    Documents,
    embed_model=embed_model,
    show_progress=True
)

Parsing nodes:   0%|          | 0/3122 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/195 [00:00<?, ?it/s]

In [18]:
# Подключаем локальную модель как движок запросов
query_engine = index.as_query_engine(llm=llm)

In [19]:
# Задаем вопрос
response = query_engine.query("Поясни, что такое производная и как она используется.")
print(response)

Производная - это математическая операция, которая определяет изменение значения функции в зависимости от её аргумента. Она показывает, как быстро меняется значение функции при изменении аргумента.

Производная часто обозначается как f'(x) и является основным инструментом в изучении динамики систем и моделировании процессов. Производная позволяет определить:

- Скорость изменения функции;
- Направление изменения функции;
- Максимальные и минимальные точки функции.

Производная широко применяется в различных областях науки и техники, включая физику, инженерное дело, экономику и медицину. Например, в физике производная может использоваться для описания движения тела под действием силы, в экономике – для анализа влияния параметров на результаты деятельности предприятия.

В контексте задачи о размножении микроорганизмов, производная функции x(t) представляет собой скорость изменения количества микроорганизмов в зависимости от времени t. Учитывая уравнение x'(t) = k * x(t), где k - коэффици

In [5]:
save_path = "./VectorStoreIndex"

In [22]:
index.storage_context.persist(persist_dir=save_path)

In [ ]:
# тест

In [13]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.graph_stores import SimpleGraphStore

import os
required_files = ["graph_store.json", "docstore.json", "index_store.json"]
if not all(os.path.exists(f"{save_path}/{f}") for f in required_files):
    raise FileNotFoundError(f"Отсутствуют необходимые файлы в {save_path}")

#  Загрузите storage context
storage_context = StorageContext.from_defaults(
    persist_dir=save_path,
    graph_store=SimpleGraphStore.from_persist_dir(save_path)
)

# Универсальный
try:
    index = load_index_from_storage(
        storage_context=storage_context,
        include_embeddings=True  # Должно соответствовать параметру при сохранении
    )
    print("Индекс успешно загружен")
except Exception as e:
    print(f"Ошибка загрузки: {e}")

Индекс успешно загружен


In [14]:
# Подключаем локальную модель как движок запросов
query_engine = index.as_query_engine(llm=llm)

In [15]:
# Задаем вопрос
response = query_engine.query("Поясни, что такое производная и как она используется.")
print(response)

Производная - это математическая операция, которая определяет изменение значения функции в зависимости от её аргумента. Она показывает, как быстро меняется значение функции при изменении аргумента. Производная часто обозначается как f'(x) и является основным инструментом в изучении динамики и изменения различных физических и экономических процессов.

Производная используется для решения многих задач, связанных с анализом функций, включая:

- Определение максимумов и минимумов функций;
- Исследование стабильности систем;
- Расчет скорости движения объектов;
- Анализ экономической эффективности проектов;
- Моделирование биологических и социальных процессов.

Производная позволяет понять, как функция ведет себя в разных точках пространства, что критически важно для принятия решений в самых разнообразных областях науки и практики.

Таким образом, производная играет ключевую роль в математическом моделировании реальности, помогая лучше понимать сложные процессы и принимать более информирова